# Análisis Integral de Frecuencias y Sensibilidad: Entregas del Río Bravo (1950-2025)
**Autor:** Diego Pallares | Braulio Silva

**Sección:** Periodos de Retorno y Sensibilidad por Modelo

Este notebook procesa las salidas de los modelos probabilísticos ajustados a las entregas históricas de los seis tributarios transfronterizos. El análisis se centra en evaluar la sensibilidad de los periodos de retorno ($T_r$) frente a cinco aproximaciones algorítmicas distintas, evaluando tanto la incertidumbre inter-modelo (GEV vs. Exponencial) como la intra-modelo (Gumbel por Momentos vs. Máxima Verosimilitud).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuración de estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 14, 'figure.titlesize': 16})

# Carpeta de salida para las gráficas
os.makedirs('resultados_sensibilidad', exist_ok=True)

# 1. Cargar el archivo real 'resultados_frecuencias'
ruta_archivo = 'resultados_frecuencias/06_periodos_retorno.csv'
df_resultados = pd.read_csv(ruta_archivo)

print("Columnas detectadas en el archivo:", df_resultados.columns.tolist())

# 2. Identificar cómo se llaman las columnas clave
col_trib = df_resultados.columns[0]
col_modelo = df_resultados.columns[1]

# 3. Transformación a Tidy Data (Formato Largo)
df_largo = df_resultados.melt(id_vars=[col_trib, col_modelo], 
                              var_name='Tr', 
                              value_name='Volumen')

# Limpieza de la columna 'Tr' 
df_largo['Tr'] = df_largo['Tr'].astype(str).str.extract(r'(\d+)').astype(int)

# Nos aseguramos de que el volumen sea numérico
df_largo['Volumen'] = pd.to_numeric(df_largo['Volumen'], errors='coerce')

print(f"\nDatos estructurados listos. Analizando {df_largo[col_trib].nunique()} tributarios y {df_largo[col_modelo].nunique()} modelos diferentes.")

Columnas detectadas en el archivo: ['Rio', 'Distribucion', 'Q_T_2', 'Q_T_5', 'Q_T_10', 'Q_T_25', 'Q_T_50', 'Q_T_100']

Datos estructurados listos. Analizando 7 tributarios y 5 modelos diferentes.


In [9]:
tributarios = df_largo[col_trib].unique()

for trib in tributarios:
    df_trib = df_largo[df_largo[col_trib] == trib].dropna(subset=['Volumen'])
    
    plt.figure(figsize=(10, 6.5))
    
    # Detectar si hay explosión de valores (para aplicar escala logarítmica inteligente)
    max_val = df_trib['Volumen'].max()
    usar_log_y = max_val > 50000  # Umbral ajustable dependiendo de tus datos
    
    sns.lineplot(
        data=df_trib, x='Tr', y='Volumen', hue=col_modelo, 
        marker='o', linewidth=2.5
    )
    
    plt.xscale('log')
    if usar_log_y:
        plt.yscale('log')
        escala_txt = "[Escala Y: Log]"
    else:
        escala_txt = "[Escala Y: Lineal]"
        
    plt.title(f'Sensibilidad y Periodos de Retorno: Río {trib} {escala_txt}', fontweight='bold')
    plt.xlabel('Periodo de Retorno ($T_r$ en Años - Escala Logarítmica)')
    plt.ylabel('Volumen de Entrega Estimado')
    
    # Asegurar que las etiquetas del eje X sean legibles
    plt.xticks([2, 5, 10, 25, 50, 100], ['2', '5', '10', '25', '50', '100'])
    
    # Mover la leyenda afuera si son muchos modelos
    plt.legend(title='Modelo/Estimador', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, which="both", ls="--", alpha=0.5)
    
    plt.tight_layout()
    nombre_limpio = str(trib).lower().replace(' ', '_').replace('.', '')
    plt.savefig(f"resultados_sensibilidad/sensibilidad_{nombre_limpio}.png", dpi=300)
    plt.close()

print("¡Procesamiento completo! Todas las gráficas se han guardado en la carpeta 'resultados_sensibilidad/'.")

¡Procesamiento completo! Todas las gráficas se han guardado en la carpeta 'resultados_sensibilidad/'.


In [10]:
# Filtramos solo para el escenario de 100 años (el más crítico para diseño)
df_100 = df_largo[df_largo['Tr'] == 100].copy()

sensibilidad_analitica = df_100.groupby(col_trib)['Volumen'].agg(
    Minimo='min',
    Maximo='max',
    Promedio='mean',
    Desviacion_Std='std'
).reset_index()

sensibilidad_analitica['Rango_Incertidumbre'] = sensibilidad_analitica['Maximo'] - sensibilidad_analitica['Minimo']

# Ordenar de mayor a menor incertidumbre para ver qué río es el más problemático
sensibilidad_analitica = sensibilidad_analitica.sort_values(by='Rango_Incertidumbre', ascending=False)

print("=========================================================================")
print("  REPORTE DE INCERTIDUMBRE Y SENSIBILIDAD CRÍTICA (Tr = 100 años)")
print("=========================================================================")
print(sensibilidad_analitica.round(2).to_string(index=False))

sensibilidad_analitica.to_csv('resultados_sensibilidad/matriz_incertidumbre_tr100.csv', index=False)

  REPORTE DE INCERTIDUMBRE Y SENSIBILIDAD CRÍTICA (Tr = 100 años)
          Rio  Minimo  Maximo  Promedio  Desviacion_Std  Rango_Incertidumbre
6 Tributarios  463.20 1268.61    754.78          306.38               805.41
       Salado  179.82  819.15    369.67          264.89               639.32
      Conchos  250.93  855.74    434.60          241.08               604.81
  San Rodrigo   70.14  650.27    206.48          249.52               580.13
    San Diego   48.35  104.22     72.06           20.37                55.87
    Escondido   19.45   74.93     35.57           22.51                55.48
        Vacas   11.93   47.06     22.43           14.26                35.13


## Conclusión del Análisis de Sensibilidad:

Al evaluar los periodos de retorno bajo 5 métodos de estimación distintos, se observa una fuerte sensibilidad algorítmica en los eventos extremos ($T_r = 100$ años). En tributarios como rio Salado, la elección del modelo provoca una incertidumbre de hasta [X unidades de volumen].Para mitigar este riesgo operativo en la planeación binacional, se cruzó este análisis de incertidumbre con las métricas de bondad de ajuste (AIC). Como resultado, se recomienda estandarizar las predicciones utilizando el modelo GEV_MLE, ya que matemáticamente representa el mejor equilibrio entre estabilidad frente a valores atípicos y representación física de la cuenca.